In [1]:
!pip install pandas numpy matplotlib scikit-learn lifelines scikit-survival openpyxl xgboost

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.0/350.0 kB 6.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 60.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 105.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 15.4 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=8366506d96f5b5b3f6224e85f761144b17a9020c083e2d6474be0306563a5cac
  Stored in directory: /root/.cache/pip/wheels/50/37/21/0a719b9d89c635e89ff24bd93b862882ad675279552013b2fb
Successfully built autograd-gamma
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [2]:
# Uncomment if first time running
# !pip install pandas numpy matplotlib scikit-learn lifelines scikit-survival openpyxl xgboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer

import xgboost as xgb

from sksurv.util import Surv
from sksurv.metrics import (
    concordance_index_censored,
    cumulative_dynamic_auc,
    integrated_brier_score
)

import warnings
warnings.filterwarnings("ignore")

# ==========================================
# Global configuration
# ==========================================

N_REPEATS = 25
RANDOM_STATE_BASE = 42
TIME_GRID = np.arange(0.25, 5.0, 0.25)

print("XGBoost Survival Pipeline Initialized")

XGBoost Survival Pipeline Initialized


In [3]:
# ==========================================
# CELL 2: Load & Preprocess Data
# ==========================================

file_path = r"/kaggle/input/datasets/swetajnathan30/umhs-dataset/Training.xlsx"

data = pd.read_excel(file_path)

# Encode categorical variables
data["Race"] = np.where(data["Race"] == "Caucasian", 1, 0)
data["Smoking"] = np.where(data["Smoking"] == "Current", 1, 0)
data["Alcohol"] = np.where(data["Alcohol"] == "Yes", 1, 0)
data["Drug"] = np.where(data["Drug"] == "Yes", 1, 0)

# Convert numeric safely
data["HGB"] = pd.to_numeric(data["HGB"], errors="coerce")
data["Duration"] = pd.to_numeric(data["Duration"], errors="coerce")
data["EndEvent"] = pd.to_numeric(data["EndEvent"], errors="coerce")
data["MAGGIC"] = pd.to_numeric(data["MAGGIC"], errors="coerce")

# Create survival columns
data["time"] = data["Duration"]
data["event"] = data["EndEvent"]

data.drop(columns=["Duration", "EndEvent"], inplace=True)

data = data.loc[:, ~data.columns.duplicated()]

print("Data Loaded. Shape:", data.shape)


Data Loaded. Shape: (343, 157)


In [5]:
# ==========================================
# CELL 3: Create D, M, DM subsets (GBSA)
# ==========================================

# Get column list
cols = list(data.columns)

# Find index positions
idx_race = cols.index("Race")
idx_c_sa = cols.index("C_SA")
idx_rvpower_s = cols.index("RVpowerIndex_S")

# ------------------------------------------
# Digital features subset (D)
# ------------------------------------------
D = pd.concat([
    data.iloc[:, idx_race:idx_c_sa],
    data[["time", "event"]]
], axis=1)

# ------------------------------------------
# MRI features subset (M)
# ------------------------------------------
M = pd.concat([
    data.iloc[:, idx_c_sa:idx_rvpower_s + 1],
    data[["time", "event"]]
], axis=1)

# ------------------------------------------
# Combined features subset (DM)
# ------------------------------------------
DM = pd.concat([
    data.iloc[:, idx_race:idx_rvpower_s + 1],
    data[["time", "event"]]
], axis=1)

# ------------------------------------------
# Remove duplicate columns (important)
# ------------------------------------------
D = D.loc[:, ~D.columns.duplicated()]
M = M.loc[:, ~M.columns.duplicated()]
DM = DM.loc[:, ~DM.columns.duplicated()]

# ------------------------------------------
# Print shapes
# ------------------------------------------
print("D:", D.shape)
print("M:", M.shape)
print("DM:", DM.shape)

D: (343, 69)
M: (343, 82)
DM: (343, 149)


In [6]:
from collections import defaultdict
from sklearn.inspection import permutation_importance
import xgboost as xgb


def xgb_variable_hunting(dataset, model_name, n_repeats=25):

    print(f"\nStarting Variable Hunting for {model_name} (XGBoost Survival)")

    feature_names = list(dataset.columns)
    feature_names.remove("time")
    feature_names.remove("event")

    freq_counter = defaultdict(int)
    importance_counter = defaultdict(float)

    X_full = dataset.drop(columns=["time", "event"])
    time_full = dataset["time"].values
    event_full = dataset["event"].values

    for rep in range(n_repeats):

        print(f"{model_name} Repeat {rep+1}/{n_repeats}")

        # XGBoost Survival Model
        model = xgb.XGBRegressor(
            objective="survival:cox",
            eval_metric="cox-nloglik",
            n_estimators=200,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bynode=0.8,
            n_jobs=-1,
            random_state=RANDOM_STATE_BASE + rep
        )

        model.fit(
            X_full,
            time_full,
            sample_weight=event_full
        )

        # Permutation importance
        perm = permutation_importance(
            model,
            X_full,
            time_full,
            n_repeats=5,
            random_state=RANDOM_STATE_BASE + rep,
            n_jobs=-1
        )

        importances = perm.importances_mean

        importance_dict = dict(zip(X_full.columns, importances))

        threshold = np.mean(importances)

        selected_features = [
            feature for feature, imp in importance_dict.items()
            if imp >= threshold
        ]

        for feature in selected_features:

            freq_counter[feature] += 1
            importance_counter[feature] += importance_dict[feature]

    avg_importance = {
        feature: importance_counter[feature] / n_repeats
        for feature in importance_counter
    }

    sorted_features = sorted(
        avg_importance.items(),
        key=lambda x: x[1],
        reverse=True
    )

    sorted_features_dict = dict(sorted_features)

    print(f"\nCompleted Variable Hunting for {model_name}")

    return {
        "frequency": dict(freq_counter),
        "importance": sorted_features_dict
    }

In [7]:
result_D = xgb_variable_hunting(D, "D", N_REPEATS)

result_M = xgb_variable_hunting(M, "M", N_REPEATS)

result_DM = xgb_variable_hunting(DM, "DM", N_REPEATS)


Starting Variable Hunting for D (XGBoost Survival)
D Repeat 1/25
D Repeat 2/25
D Repeat 3/25
D Repeat 4/25
D Repeat 5/25
D Repeat 6/25
D Repeat 7/25
D Repeat 8/25
D Repeat 9/25
D Repeat 10/25
D Repeat 11/25
D Repeat 12/25
D Repeat 13/25
D Repeat 14/25
D Repeat 15/25
D Repeat 16/25
D Repeat 17/25
D Repeat 18/25
D Repeat 19/25
D Repeat 20/25
D Repeat 21/25
D Repeat 22/25
D Repeat 23/25
D Repeat 24/25
D Repeat 25/25

Completed Variable Hunting for D

Starting Variable Hunting for M (XGBoost Survival)
M Repeat 1/25
M Repeat 2/25
M Repeat 3/25
M Repeat 4/25
M Repeat 5/25
M Repeat 6/25
M Repeat 7/25
M Repeat 8/25
M Repeat 9/25
M Repeat 10/25
M Repeat 11/25
M Repeat 12/25
M Repeat 13/25
M Repeat 14/25
M Repeat 15/25
M Repeat 16/25
M Repeat 17/25
M Repeat 18/25
M Repeat 19/25
M Repeat 20/25
M Repeat 21/25
M Repeat 22/25
M Repeat 23/25
M Repeat 24/25
M Repeat 25/25

Completed Variable Hunting for M

Starting Variable Hunting for DM (XGBoost Survival)
DM Repeat 1/25
DM Repeat 2/25
DM Repeat 3/2

In [9]:
print("\nTop features in D:")
print(list(result_D["importance"].items())[:10])

print("\nTop features in M:")
print(list(result_M["importance"].items())[:10])

print("\nTop features in DM:")
print(list(result_DM["importance"].items())[:10])


Top features in D:
[('LVIDd_D', np.float64(5.591133690937792e+27)), ('EAr_D', np.float64(1.8990107528892532e+27)), ('RVESV_D', np.float64(7.48256208777502e+26)), ('LVIDs_D', np.float64(6.603814968302148e+26)), ('Age', np.float64(5.805925403473679e+26)), ('HGB', np.float64(5.548975844824722e+26)), ('PAC_D', np.float64(2.621276105673134e+26)), ('MVr_D', np.float64(2.558485583281717e+26)), ('HTN', np.float64(2.3329818641599344e+26)), ('Race', np.float64(8.155775668374892e+25))]

Top features in M:
[('Amref_SEP', np.float64(4.07821269032934e+28)), ('k_pas_LV', np.float64(3.1757492696478465e+28)), ('StressSEP_S', np.float64(1.6869979609046319e+28)), ('SBP_S', np.float64(1.6039400964156817e+28)), ('CO_S', np.float64(8.517579109536971e+27)), ('k_act_LV', np.float64(6.787481597346463e+27)), ('Hed_LW_S', np.float64(6.317214958707673e+27)), ('Vw_RV', np.float64(4.046520479219401e+27)), ('LVMD_S', np.float64(1.533275892315927e+27)), ('R_RA', np.float64(1.0852249668342578e+27))]

Top features in 

In [10]:
result_D
result_M
result_DM

{'frequency': {'Race': 25,
  'Age': 22,
  'Gender': 25,
  'BMI': 19,
  'Smoking': 25,
  'Alcohol': 25,
  'Drug': 25,
  'HGB': 25,
  'Creatinine': 22,
  'Weight': 16,
  'Height': 24,
  'BSA': 8,
  'Usage of ERA': 25,
  'Usage of Diuretic': 25,
  'O2 Supp': 23,
  'HTN': 25,
  'PH': 25,
  'CAD': 25,
  'AF': 25,
  'HLD': 25,
  'DM': 25,
  'COPD': 24,
  'CTD': 14,
  'ICDorPacemaker': 25,
  'Renal failure': 25,
  "Septal E/e'": 20,
  "Lateral E/e'": 21,
  'SBP_D': 12,
  'RAPmax_D': 10,
  'RVEDP_D': 23,
  'minRVP_D': 25,
  'PCWP_D': 22,
  'LVIDs_D': 25,
  'LVIDd_D': 22,
  'LVm_D': 23,
  'RVm_D': 25,
  'EF_D': 17,
  'LVEDV_D': 25,
  'LVESV_D': 25,
  'RVEDV_D': 22,
  'RVESV_D': 21,
  'Hed_LW_D': 25,
  'Hed_SW_D': 22,
  'LAVmax_D': 16,
  'LVESP_D': 21,
  'LVEDP_D': 20,
  'minLVP_D': 23,
  'MVr_D': 25,
  'TVr_D': 24,
  'PVr_D': 25,
  'AVr_D': 25,
  'EAr_D': 24,
  'PAC_D': 24,
  'PVR_D': 21,
  'RVSWI_D': 11,
  'LVpower_D': 25,
  'RVpower_D': 24,
  'C_SV': 24,
  'C_PV': 22,
  'k_act_LV': 21,
  'k_a

In [11]:
def extract_feature_groups(result, n_repeats=N_REPEATS):

    freq = result["frequency"]
    importance = result["importance"]

    # Mandatory features: ≥80% frequency
    mandatory = [
        feature for feature, count in freq.items()
        if count >= 0.8 * n_repeats
    ]

    # Valid features: ≥20% frequency (more stable threshold)
    valid = [
        feature for feature, count in freq.items()
        if count >= 0.2 * n_repeats
    ]

    # Sort valid features by importance
    sorted_valid = {
        feature: importance[feature]
        for feature in valid
        if feature in importance
    }

    sorted_valid = dict(
        sorted(sorted_valid.items(),
               key=lambda x: x[1],
               reverse=True)
    )

    return mandatory, valid, sorted_valid


# Apply to datasets
mandatory_D, valid_D, sorted_valid_D = extract_feature_groups(result_D)
mandatory_M, valid_M, sorted_valid_M = extract_feature_groups(result_M)
mandatory_DM, valid_DM, sorted_valid_DM = extract_feature_groups(result_DM)


# Print summary

print("Mandatory D:", len(mandatory_D))
print("Valid D:", len(valid_D))


print("Mandatory M:", len(mandatory_M))
print("Valid M:", len(valid_M))


print("Mandatory DM:", len(mandatory_DM))
print("Valid DM:", len(valid_DM))

Mandatory D: 45
Valid D: 58
Mandatory M: 46
Valid M: 74
Mandatory DM: 94
Valid DM: 139


In [12]:
import xgboost as xgb
from sksurv.metrics import concordance_index_censored


def train_xgb_model(dataset, features, random_state):

    X = dataset[features]
    time = dataset["time"].values
    event = dataset["event"].values.astype(bool)

    model = xgb.XGBRegressor(
        objective="survival:cox",
        eval_metric="cox-nloglik",
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bynode=0.8,
        n_jobs=-1,
        random_state=random_state
    )

    model.fit(
        X,
        time,
        sample_weight=event.astype(int)
    )

    risk_scores = model.predict(X)

    cindex = concordance_index_censored(
        event,
        time,
        risk_scores
    )[0]

    return cindex

In [13]:
def find_best_features_xgb(dataset, sorted_valid_features,
                           mandatory_features,
                           model_name,
                           max_additional=12,
                           repeats=10):

    print(f"\nSelecting best features for {model_name} (XGBoost Survival)")

    selected = mandatory_features.copy()

    remaining = list(sorted_valid_features.keys())
    remaining = [f for f in remaining if f not in selected]

    history = []

    # Evaluate mandatory features first
    scores = []

    for rep in range(repeats):

        cindex = train_xgb_model(
            dataset,
            selected,
            RANDOM_STATE_BASE + rep
        )

        scores.append(cindex)

    best_score = np.mean(scores)

    history.append((len(selected), best_score, selected.copy()))

    print(f"Initial features: {len(selected)} | C-index: {best_score:.4f}")

    # Forward selection
    for i in range(max_additional):

        best_feature = None
        best_feature_score = best_score

        for feature in remaining:

            test_features = selected + [feature]

            scores = []

            for rep in range(repeats):

                cindex = train_xgb_model(
                    dataset,
                    test_features,
                    RANDOM_STATE_BASE + rep
                )

                scores.append(cindex)

            mean_score = np.mean(scores)

            if mean_score > best_feature_score:

                best_feature_score = mean_score
                best_feature = feature

        if best_feature is None:
            break

        selected.append(best_feature)
        remaining.remove(best_feature)
        best_score = best_feature_score

        history.append(
            (len(selected), best_score, selected.copy())
        )

        print(
            f"Added: {best_feature} | Total: {len(selected)} | "
            f"C-index: {best_score:.4f}"
        )

    return selected, history

In [14]:
selected_D, history_D = find_best_features_xgb(
    D,
    sorted_valid_D,
    mandatory_D,
    "D"
)

selected_M, history_M = find_best_features_xgb(
    M,
    sorted_valid_M,
    mandatory_M,
    "M"
)

selected_DM, history_DM = find_best_features_xgb(
    DM,
    sorted_valid_DM,
    mandatory_DM,
    "DM"
)

print("\nFinal selected features:")
print("D:", len(selected_D))
print("M:", len(selected_M))
print("DM:", len(selected_DM))


Selecting best features for D (XGBoost Survival)
Initial features: 45 | C-index: 0.6424
Added: EAr_D | Total: 46 | C-index: 0.6569
Added: NYHA class | Total: 47 | C-index: 0.6658
Added: PASP_D | Total: 48 | C-index: 0.6759
Added: Septal E/e' | Total: 49 | C-index: 0.6763

Selecting best features for M (XGBoost Survival)
Initial features: 46 | C-index: 0.6495
Added: LVm_S | Total: 47 | C-index: 0.6609
Added: R_m_o | Total: 48 | C-index: 0.6686

Selecting best features for DM (XGBoost Survival)
Initial features: 94 | C-index: 0.6527
Added: PVR_S | Total: 95 | C-index: 0.6553
Added: Vw_LV | Total: 96 | C-index: 0.6598
Added: NYHA class | Total: 97 | C-index: 0.6680
Added: TVr_S | Total: 98 | C-index: 0.6753
Added: BSA | Total: 99 | C-index: 0.6757
Added: RVSWI_D | Total: 100 | C-index: 0.6800
Added: SVR_D | Total: 101 | C-index: 0.6821
Added: PASP_D | Total: 102 | C-index: 0.6850
Added: C_SA | Total: 103 | C-index: 0.6875
Added: Amref_SEP | Total: 104 | C-index: 0.6888
Added: SBP_D | Tot

In [15]:
from sklearn.model_selection import train_test_split

def split_dataset_xgb(dataset, selected_features, model_name, test_size=0.3):

    print(f"\nSplitting dataset for {model_name}")

    X = dataset[selected_features]
    time = dataset["time"].values
    event = dataset["event"].values

    X_train, X_test, time_train, time_test, event_train, event_test = train_test_split(
        X,
        time,
        event,
        test_size=test_size,
        random_state=42
    )

    print(f"{model_name} Train shape:", X_train.shape)
    print(f"{model_name} Test shape:", X_test.shape)

    return (
        X_train, X_test,
        time_train, time_test,
        event_train, event_test
    )


# Split datasets
X_train_D, X_test_D, time_train_D, time_test_D, event_train_D, event_test_D = \
    split_dataset_xgb(D, selected_D, "D")

X_train_M, X_test_M, time_train_M, time_test_M, event_train_M, event_test_M = \
    split_dataset_xgb(M, selected_M, "M")

X_train_DM, X_test_DM, time_train_DM, time_test_DM, event_train_DM, event_test_DM = \
    split_dataset_xgb(DM, selected_DM, "DM")


Splitting dataset for D
D Train shape: (240, 49)
D Test shape: (103, 49)

Splitting dataset for M
M Train shape: (240, 48)
M Test shape: (103, 48)

Splitting dataset for DM
DM Train shape: (240, 106)
DM Test shape: (103, 106)


In [18]:
from sklearn.model_selection import KFold, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sksurv.metrics import concordance_index_censored
import xgboost as xgb
import numpy as np
import pandas as pd


def tune_xgb_cv(X_train, time_train, event_train, model_name, n_splits=5):

    print(f"\nHyperparameter tuning with CV for {model_name} (XGBoost Survival)")

    # ==========================================
    # Scale features (VERY IMPORTANT)
    # ==========================================

    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )

    # ==========================================
    # Parameter grid (stable range)
    # ==========================================

    param_grid = {
        "n_estimators": [200, 400],
        "max_depth": [2, 3, 4],
        "learning_rate": [0.03, 0.05],
        "subsample": [0.7, 0.9],
        "colsample_bynode": [0.7, 0.9]
    }

    kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    best_score = -1
    best_params = None

    # ==========================================
    # Grid Search
    # ==========================================

    for params in ParameterGrid(param_grid):

        fold_scores = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_scaled)):

            X_fold_train = X_train_scaled.iloc[train_idx]
            X_fold_val   = X_train_scaled.iloc[val_idx]

            time_fold_train = time_train[train_idx]
            time_fold_val   = time_train[val_idx]

            event_fold_train = event_train[train_idx]
            event_fold_val   = event_train[val_idx]

            # ==========================================
            # Stable XGBoost survival model
            # ==========================================

            model = xgb.XGBRegressor(

                objective="survival:cox",
                eval_metric="cox-nloglik",

                max_delta_step=1,    # prevents overflow
                reg_lambda=1.0,      # L2 regularization
                reg_alpha=0.1,       # L1 regularization

                n_jobs=-1,
                random_state=42,

                **params
            )

            # Train
            model.fit(
                X_fold_train,
                time_fold_train,
                sample_weight=event_fold_train
            )

            # Predict risk
            risk_scores = model.predict(X_fold_val)

            # ==========================================
            # Fix infinity / overflow
            # ==========================================

            risk_scores = np.nan_to_num(
                risk_scores,
                nan=0.0,
                posinf=1e6,
                neginf=-1e6
            )

            risk_scores = np.clip(
                risk_scores,
                -1e6,
                1e6
            )

            # Compute C-index
            cindex = concordance_index_censored(
                event_fold_val.astype(bool),
                time_fold_val,
                risk_scores
            )[0]

            fold_scores.append(cindex)

        mean_score = np.mean(fold_scores)
        std_score  = np.std(fold_scores)

        print(f"{params} → Mean CV C-index: {mean_score:.4f} (±{std_score:.4f})")

        if mean_score > best_score:

            best_score = mean_score
            best_params = params

    print(f"\nBest params for {model_name}:")
    print(best_params)
    print(f"Best CV C-index: {best_score:.4f}")

    return best_params

In [19]:
best_params_D = tune_xgb_cv(
    X_train_D,
    time_train_D,
    event_train_D,
    "D"
)

best_params_M = tune_xgb_cv(
    X_train_M,
    time_train_M,
    event_train_M,
    "M"
)

best_params_DM = tune_xgb_cv(
    X_train_DM,
    time_train_DM,
    event_train_DM,
    "DM"
)


Hyperparameter tuning with CV for D (XGBoost Survival)
{'colsample_bynode': 0.7, 'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 200, 'subsample': 0.7} → Mean CV C-index: 0.5397 (±0.0393)
{'colsample_bynode': 0.7, 'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 200, 'subsample': 0.9} → Mean CV C-index: 0.5504 (±0.0529)
{'colsample_bynode': 0.7, 'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 400, 'subsample': 0.7} → Mean CV C-index: 0.5000 (±0.0000)
{'colsample_bynode': 0.7, 'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 400, 'subsample': 0.9} → Mean CV C-index: 0.5000 (±0.0000)
{'colsample_bynode': 0.7, 'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.7} → Mean CV C-index: 0.5424 (±0.0371)
{'colsample_bynode': 0.7, 'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.9} → Mean CV C-index: 0.5560 (±0.0534)
{'colsample_bynode': 0.7, 'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 400, 'subsample': 0.7} 

In [20]:
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import numpy as np


def train_and_evaluate_test_xgb(
    X_train, time_train, event_train,
    X_test, time_test, event_test,
    best_params, model_name):

    print(f"\nFinal XGBoost Survival training and testing for {model_name}")

    # ==========================================
    # Scale features (CRITICAL)
    # ==========================================

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # ==========================================
    # Train model
    # ==========================================

    model = xgb.XGBRegressor(

        objective="survival:cox",
        eval_metric="cox-nloglik",

        max_delta_step=1,
        reg_lambda=1.0,
        reg_alpha=0.1,

        n_jobs=-1,
        random_state=42,

        **best_params
    )

    model.fit(
        X_train_scaled,
        time_train,
        sample_weight=event_train
    )

    # ==========================================
    # Predict risk scores
    # ==========================================

    risk_scores = model.predict(X_test_scaled)

    # Fix overflow
    risk_scores = np.nan_to_num(
        risk_scores,
        nan=0.0,
        posinf=1e6,
        neginf=-1e6
    )

    risk_scores = np.clip(risk_scores, -1e6, 1e6)

    # ==========================================
    # C-index
    # ==========================================

    cindex = concordance_index_censored(
        event_test.astype(bool),
        time_test,
        risk_scores
    )[0]

    # ==========================================
    # Create structured arrays for sksurv metrics
    # ==========================================

    y_train_struct = Surv.from_arrays(event_train.astype(bool), time_train)
    y_test_struct  = Surv.from_arrays(event_test.astype(bool), time_test)

    # ==========================================
    # Time grid
    # ==========================================

    max_time = min(time_train.max(), time_test.max())

    valid_times = np.linspace(
        time_test.min() + 1e-6,
        max_time * 0.99,
        20
    )

    # ==========================================
    # iAUC
    # ==========================================

    auc_times, auc_scores = cumulative_dynamic_auc(
        y_train_struct,
        y_test_struct,
        risk_scores,
        valid_times
    )

    iauc = np.mean(auc_scores)

    # ==========================================
    # Brier score
    # ==========================================

    # Convert risk score to survival probability approximation
    surv_matrix = np.exp(-np.outer(risk_scores, valid_times / valid_times.max()))

    brier = integrated_brier_score(
        y_train_struct,
        y_test_struct,
        surv_matrix,
        valid_times
    )

    # ==========================================
    # Print results
    # ==========================================

    print(f"\n{model_name} TEST Results:")
    print(f"C-index: {cindex:.4f}")
    print(f"iAUC:    {iauc:.4f}")
    print(f"Brier:   {brier:.4f}")

    return model, scaler, cindex, iauc, brier

In [21]:
model_D, scaler_D, cindex_D, iauc_D, brier_D = train_and_evaluate_test_xgb(
    X_train_D, time_train_D, event_train_D,
    X_test_D, time_test_D, event_test_D,
    best_params_D,
    "D"
)

model_M, scaler_M, cindex_M, iauc_M, brier_M = train_and_evaluate_test_xgb(
    X_train_M, time_train_M, event_train_M,
    X_test_M, time_test_M, event_test_M,
    best_params_M,
    "M"
)

model_DM, scaler_DM, cindex_DM, iauc_DM, brier_DM = train_and_evaluate_test_xgb(
    X_train_DM, time_train_DM, event_train_DM,
    X_test_DM, time_test_DM, event_test_DM,
    best_params_DM,
    "DM"
)


Final XGBoost Survival training and testing for D

D TEST Results:
C-index: 0.5741
iAUC:    0.5574
Brier:   0.7052

Final XGBoost Survival training and testing for M

M TEST Results:
C-index: 0.5036
iAUC:    0.4826
Brier:   0.7052

Final XGBoost Survival training and testing for DM

DM TEST Results:
C-index: 0.5331
iAUC:    0.5081
Brier:   0.7052


In [22]:
test_data = pd.read_excel("/kaggle/input/datasets/swetajnathan30/umhs-dataset/Testing.xlsx")

# Apply SAME preprocessing
test_data["Race"] = np.where(test_data["Race"] == "Caucasian", 1, 0)
test_data["Smoking"] = np.where(test_data["Smoking"] == "Current", 1, 0)
test_data["Alcohol"] = np.where(test_data["Alcohol"] == "Yes", 1, 0)
test_data["Drug"] = np.where(test_data["Drug"] == "Yes", 1, 0)

test_data["time"] = test_data["Duration"]
test_data["event"] = test_data["EndEvent"]

test_data.drop(columns=["Duration", "EndEvent"], inplace=True)

In [23]:
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import numpy as np
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored, cumulative_dynamic_auc, integrated_brier_score


def train_and_evaluate_external_xgb(train_dataset, test_dataset,
                                    selected_features, best_params,
                                    model_name):

    print(f"\nFINAL XGBoost Survival Training and External Evaluation: {model_name}")

    # ==========================================
    # Prepare training data
    # ==========================================

    X_train = train_dataset[selected_features]
    time_train = train_dataset["time"].values
    event_train = train_dataset["event"].values

    # ==========================================
    # Prepare external test data
    # ==========================================

    X_test = test_dataset[selected_features]
    time_test = test_dataset["time"].values
    event_test = test_dataset["event"].values

    # ==========================================
    # Scale features (IMPORTANT)
    # ==========================================

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # ==========================================
    # Train model on FULL training dataset
    # ==========================================

    model = xgb.XGBRegressor(

        objective="survival:cox",
        eval_metric="cox-nloglik",

        max_delta_step=1,
        reg_lambda=1.0,
        reg_alpha=0.1,

        n_jobs=-1,
        random_state=42,

        **best_params
    )

    model.fit(
        X_train_scaled,
        time_train,
        sample_weight=event_train
    )

    # ==========================================
    # Predict risk scores
    # ==========================================

    risk_scores = model.predict(X_test_scaled)

    # Fix overflow
    risk_scores = np.nan_to_num(
        risk_scores,
        nan=0.0,
        posinf=1e6,
        neginf=-1e6
    )

    risk_scores = np.clip(risk_scores, -1e6, 1e6)

    # ==========================================
    # C-index
    # ==========================================

    cindex = concordance_index_censored(
        event_test.astype(bool),
        time_test,
        risk_scores
    )[0]

    # ==========================================
    # Structured arrays for survival metrics
    # ==========================================

    y_train_struct = Surv.from_arrays(event_train.astype(bool), time_train)
    y_test_struct  = Surv.from_arrays(event_test.astype(bool), time_test)

    # ==========================================
    # Dynamic time grid
    # ==========================================

    max_time = min(time_train.max(), time_test.max())

    valid_times = np.linspace(
        time_test.min() + 1e-6,
        max_time * 0.99,
        20
    )

    # ==========================================
    # Time-dependent AUC (iAUC)
    # ==========================================

    auc_times, auc_scores = cumulative_dynamic_auc(
        y_train_struct,
        y_test_struct,
        risk_scores,
        valid_times
    )

    iauc = np.mean(auc_scores)

    # ==========================================
    # Integrated Brier Score
    # ==========================================

    # Survival probability approximation
    surv_matrix = np.exp(-np.outer(risk_scores, valid_times / valid_times.max()))

    brier = integrated_brier_score(
        y_train_struct,
        y_test_struct,
        surv_matrix,
        valid_times
    )

    # ==========================================
    # Print results
    # ==========================================

    print(f"\n{model_name} External Results:")
    print(f"C-index: {cindex:.4f}")
    print(f"iAUC:    {iauc:.4f}")
    print(f"Brier:   {brier:.4f}")

    return model, scaler, cindex, iauc, brier

In [24]:
model_D, scaler_D, cindex_D, iauc_D, brier_D = train_and_evaluate_external_xgb(
    data, test_data,
    selected_D,
    best_params_D,
    "D"
)

model_M, scaler_M, cindex_M, iauc_M, brier_M = train_and_evaluate_external_xgb(
    data, test_data,
    selected_M,
    best_params_M,
    "M"
)

model_DM, scaler_DM, cindex_DM, iauc_DM, brier_DM = train_and_evaluate_external_xgb(
    data, test_data,
    selected_DM,
    best_params_DM,
    "DM"
)


FINAL XGBoost Survival Training and External Evaluation: D

D External Results:
C-index: 0.5568
iAUC:    0.5420
Brier:   0.7223

FINAL XGBoost Survival Training and External Evaluation: M

M External Results:
C-index: 0.4687
iAUC:    0.4713
Brier:   0.7223

FINAL XGBoost Survival Training and External Evaluation: DM

DM External Results:
C-index: 0.5432
iAUC:    0.5166
Brier:   0.7223
